In [1]:
import datar.all as dr
from datar import f

import polars as pl
from polars import col as c

from pathlib import Path
import warnings

warnings.filterwarnings("ignore")
pl.Config.set_tbl_cell_alignment("CENTER")

data_dir = next(Path("/home/").glob("**/datar-polars/data"))

In [2]:
tb = dr.tibble(
    x = [1, 2, 3],
    y = ["a", "b", "c"],
    z = [True, False, True]
)
print(tb)

shape: (3, 3)
┌─────┬─────┬───────┐
│  x  ┆  y  ┆   z   │
│ --- ┆ --- ┆  ---  │
│ i64 ┆ str ┆  bool │
╞═════╪═════╪═══════╡
│  1  ┆  a  ┆  true │
│  2  ┆  b  ┆ false │
│  3  ┆  c  ┆  true │
└─────┴─────┴───────┘


In [3]:
df = pl.DataFrame({
    'A': [1, 2, 3],
    'B': ['x', 'y', 'z'],
    'C': [True, False, True]
})
print(df)

shape: (3, 3)
┌─────┬─────┬───────┐
│  A  ┆  B  ┆   C   │
│ --- ┆ --- ┆  ---  │
│ i64 ┆ str ┆  bool │
╞═════╪═════╪═══════╡
│  1  ┆  x  ┆  true │
│  2  ┆  y  ┆ false │
│  3  ┆  z  ┆  true │
└─────┴─────┴───────┘


# <span style="color:#1E90FF">1. "f" expression</span>

## "f" expression syntax allows for more intuitive and readable DataFrame operations,
## it liberates us from typing the DataFrame name repeatedly (like df[df['A'] > 1])

In [4]:
filtered_tb = tb >> dr.filter_(f.x > 1) # Use ``f.col_name`` to access column ``col_name`` conveniently 
print(filtered_tb)

shape: (2, 3)
┌─────┬─────┬───────┐
│  x  ┆  y  ┆   z   │
│ --- ┆ --- ┆  ---  │
│ i64 ┆ str ┆  bool │
╞═════╪═════╪═══════╡
│  2  ┆  b  ┆ false │
│  3  ┆  c  ┆  true │
└─────┴─────┴───────┘


In [5]:
selected_tb = tb >> dr.select(f['y'], f['z']) # Can also work with subscript style like ``f['col_name']``
print(selected_tb)

shape: (3, 2)
┌─────┬───────┐
│  y  ┆   z   │
│ --- ┆  ---  │
│ str ┆  bool │
╞═════╪═══════╡
│  a  ┆  true │
│  b  ┆ false │
│  c  ┆  true │
└─────┴───────┘


In [6]:
selected_tb2 = tb >> dr.select(f[['x', 'z']])
print(selected_tb2)

shape: (3, 2)
┌─────┬───────┐
│  x  ┆   z   │
│ --- ┆  ---  │
│ i64 ┆  bool │
╞═════╪═══════╡
│  1  ┆  true │
│  2  ┆ false │
│  3  ┆  true │
└─────┴───────┘


In [7]:
# Compatible with Polars DataFrame

filtered_df = df >> dr.filter_(f["A"] > 1) # Cannot use ``f.A`` for Polars DataFrame because it is not supported natively by Polars
print(filtered_df)

shape: (2, 3)
┌─────┬─────┬───────┐
│  A  ┆  B  ┆   C   │
│ --- ┆ --- ┆  ---  │
│ i64 ┆ str ┆  bool │
╞═════╪═════╪═══════╡
│  2  ┆  y  ┆ false │
│  3  ┆  z  ┆  true │
└─────┴─────┴───────┘


# <span style="color:#1E90FF">2. Pipe Operator ">>"</span>
## Pipe Operator ">>" allows chaining multiple DataFrame operations in a readable manner.

In [17]:
tb_pokemon = (
    dr.tibble(pl.read_csv(data_dir/"pokemon.csv"))
    >> dr.rename_with(lambda col: col.strip().replace(" ", "_").replace(".", "")) # Clean column names
    >> dr.select(~f["#"]) # Drop the "#" column
    >> dr.mutate(
        Type_1 = f.Type_1.cast(pl.Categorical),      # convert to category (pandas style)
        Type_2 = dr.as_factor(f.Type_2),           # convert to category (datar style)
        Generation = dr.as_ordered(f.Generation),  # convert to ordered category (datar style)
        Legendary = dr.as_logical(f.Legendary)     # convert to boolean (datar style)
    )
)

print(
    tb_pokemon
    >> dr.slice_head(n=5)
)
'''The dr.as_ordered() does not work as expected, it returns ``cat``, not ``enum``'''

shape: (5, 12)
┌───────────────────────┬────────┬────────┬───────┬───┬────────┬───────┬────────────┬───────────┐
│          Name         ┆ Type_1 ┆ Type_2 ┆ Total ┆ … ┆ Sp_Def ┆ Speed ┆ Generation ┆ Legendary │
│          ---          ┆   ---  ┆   ---  ┆  ---  ┆   ┆   ---  ┆  ---  ┆     ---    ┆    ---    │
│          str          ┆   cat  ┆   cat  ┆  i64  ┆   ┆   i64  ┆  i64  ┆     cat    ┆    bool   │
╞═══════════════════════╪════════╪════════╪═══════╪═══╪════════╪═══════╪════════════╪═══════════╡
│       Bulbasaur       ┆  Grass ┆ Poison ┆  318  ┆ … ┆   65   ┆   45  ┆      1     ┆   false   │
│        Ivysaur        ┆  Grass ┆ Poison ┆  405  ┆ … ┆   80   ┆   60  ┆      1     ┆   false   │
│        Venusaur       ┆  Grass ┆ Poison ┆  525  ┆ … ┆   100  ┆   80  ┆      1     ┆   false   │
│ VenusaurMega Venusaur ┆  Grass ┆ Poison ┆  625  ┆ … ┆   120  ┆   80  ┆      1     ┆   false   │
│       Charmander      ┆  Fire  ┆  null  ┆  309  ┆ … ┆   50   ┆   65  ┆      1     ┆   false   │
└────

'The dr.as_ordered() does not work as expected, it returns ``cat``, not ``enum``'

# <span style="color:#1E90FF">3. Functions that support "f" expression and Pipe Operator ">>" ">>"</span>
### dr.slice_head(), dr.slice_tail(), dr.glimpse(), dr.rename(),
### dr.inner_join(), dr.left_join(), dr.right_join(), dr.full_join(), dr.anti_join(), dr.semi_join(),
### dr.pivot_wider(), dr.pivot_longer()
### dr.select(), dr.pull(), dr.relocate()
### dr.slice_(), dr.slice_min(), dr.slice_max()
### dr.filter_()
### dr.add_column(), dr.add_row(), dr.mutate()
### dr.summarise()
### dr.pipe()
### dr.extract(), dr.separate(), dr.separate_rows(), dr.unite()